# Notebook 02 — Born Rule and Quantum Measurement

本 Notebook 将量子态的概率振幅转化为测量概率，并用 NumPy 模拟重复测量。

**学习目标**

- 理解 Born 定则：测量概率等于概率振幅模的平方
- 区分“单次测量结果”和“重复测量的统计频率”
- 使用 `numpy.random.Generator.choice` 模拟量子测量
- 检查概率非负、总和为 1，并比较理论概率与数值频率

建议从上到下逐个运行单元格。先完成“练习”部分，再看后面的参考实现。

## 1. 必要理论：Born 定则

单量子比特在计算基下可以写成

$$
|\\psi\\rangle = \\alpha|0\\rangle + \\beta|1\\rangle,
\\qquad
|\\psi\\rangle =
\\begin{pmatrix}
\\alpha \\\\
\\beta
\\end{pmatrix}.
$$

其中 $\\alpha,\\beta$ 是复数振幅。若量子态已归一化，则

$$
|\\alpha|^2 + |\\beta|^2 = 1.
$$

在计算基 $\\{|0\\rangle, |1\\rangle\\}$ 下进行测量，Born 定则给出

$$
P(0)=|\\langle 0|\\psi\\rangle|^2=|\\alpha|^2,
\\qquad
P(1)=|\\langle 1|\\psi\\rangle|^2=|\\beta|^2.
$$

**重要区别：** 一次测量只会得到一个经典结果（0 或 1），不会同时得到两个概率。概率描述的是在许多次相同制备、相同测量的实验中，各结果出现的比例趋势。

## 2. 本题给定的量子态

考虑

$$
|\\psi\\rangle =
\\frac{1}{2}|0\\rangle
+ \\frac{i\\sqrt{3}}{2}|1\\rangle
=
\\begin{pmatrix}
1/2 \\\\
i\\sqrt{3}/2
\\end{pmatrix}.
$$

因此理论上

$$
P(0)=\\left|\\frac12\\right|^2=\\frac14,
\\qquad
P(1)=\\left|\\frac{i\\sqrt3}{2}\\right|^2=\\frac34.
$$

请注意：复数振幅的相位会影响量子态的性质，但在这次计算基测量的概率中，取的是振幅的模平方。

## 3. 练习任务（先自己完成）

请用 NumPy 完成以下任务：

1. 将给定量子态存为复数数组 `psi`。
2. 检查它是否归一化，即 `np.vdot(psi, psi)` 是否为 1。
3. 根据 Born 定则计算 `p0` 和 `p1`，并组成数组 `probabilities`。
4. 检查两个概率都非负，且总和为 1。
5. 使用 10,000 次独立测量模拟，保存每次结果（0 或 1）。
6. 统计 0 和 1 分别出现多少次，并计算实验频率。
7. 比较实验频率与理论概率。

**提示**

- 模复数：`np.abs(psi) ** 2`
- 可重复的随机数生成器：`rng = np.random.default_rng(seed=2026)`
- 按给定概率抽样：`rng.choice([0, 1], size=shots, p=probabilities)`
- 统计 0 和 1：`np.bincount(outcomes, minlength=2)`

实验频率通常不会刚好等于理论概率。不要用浮点数严格相等来判断统计结果。

In [ ]:
# TODO 1: 导入 NumPy
import numpy as np

# TODO 2: 定义量子态 psi
psi = None

# TODO 3: 计算理论测量概率
probabilities = None

# TODO 4: 检查归一化与概率条件


# TODO 5: 模拟 10,000 次独立测量
shots = 10_000
rng = np.random.default_rng(seed=2026)
outcomes = None

# TODO 6: 统计次数与实验频率
counts = None
frequencies = None

# TODO 7: 输出理论概率、实验频率与偏差


## 4. 参考实现

先尝试完成上一格，再运行下面的参考实现。

In [ ]:
import numpy as np

# 1. 定义归一化的单量子比特状态
psi = np.array([1 / 2, 1j * np.sqrt(3) / 2], dtype=np.complex128)

# 2. 验证量子态归一化
norm_squared = np.vdot(psi, psi).real
print("<psi|psi> =", norm_squared)
assert np.isclose(norm_squared, 1.0)

# 3. Born 定则：概率等于振幅模的平方
probabilities = np.abs(psi) ** 2
p0, p1 = probabilities

print("Theoretical P(0) =", p0)
print("Theoretical P(1) =", p1)

# 4. 检查概率条件
assert np.all(probabilities >= 0.0)
assert np.isclose(probabilities.sum(), 1.0)

# 5. 重复进行 10,000 次独立测量
shots = 10_000
rng = np.random.default_rng(seed=2026)
outcomes = rng.choice([0, 1], size=shots, p=probabilities)

# 6. 统计测量结果
counts = np.bincount(outcomes, minlength=2)
frequencies = counts / shots

print("\\nNumber of shots =", shots)
print("Counts [0, 1] =", counts)
print("Observed frequencies =", frequencies)
print("Absolute deviations =", np.abs(frequencies - probabilities))

assert counts.sum() == shots
assert np.all(frequencies >= 0.0)
assert np.isclose(frequencies.sum(), 1.0)

## 5. 如何理解模拟结果？

使用固定随机种子 `2026`，上面的代码应得到：

| 结果 | 理论概率 | 模拟次数 | 实验频率 |
|---|---:|---:|---:|
| 0 | 0.25 | 2447 | 0.2447 |
| 1 | 0.75 | 7553 | 0.7553 |

实验频率与理论概率很接近，但并不完全相等。这是有限次随机抽样的正常现象。

增加测量次数时，实验频率通常会更接近理论概率；这体现了大数定律的统计思想。注意，这并不表示每一次新增实验都必然让误差单调减小。

## 6. 一个小改动：改变测量次数

试着把 `shots` 改成 100、1,000、100,000，观察实验频率与理论概率的偏差如何变化。

可以记录不同测量次数下的结果：

| shots | frequency of 0 | frequency of 1 | deviation |
|---:|---:|---:|---:|
| 100 | 自己填写 | 自己填写 | 自己填写 |
| 1,000 | 自己填写 | 自己填写 | 自己填写 |
| 10,000 | 自己填写 | 自己填写 | 自己填写 |
| 100,000 | 自己填写 | 自己填写 | 自己填写 |

**思考：** 为什么增加测量次数后，实验频率一般会更稳定？为什么单次测量仍然无法预测一定得到 0 还是 1？

## 7. 常见错误与本题边界

1. **把振幅直接当成概率。** 概率是 $|\\alpha|^2$，不是 $\\alpha$。
2. **对复数直接平方。** 应使用 `np.abs(psi) ** 2`，而不是 `psi ** 2`。
3. **忘记归一化。** 抽样概率必须非负且总和为 1；本题给定的状态已经归一化。
4. **认为频率必须等于概率。** 有限样本存在统计涨落。
5. **把多次测量误解为同一量子态连续测量。** 这里的每个 shot 都假设重新制备同一个初态，再测量一次。

本 Notebook 只模拟计算基下的单量子比特测量结果。它用经典伪随机抽样复现 Born 定则给出的统计分布，并没有模拟真实硬件，也没有研究测量后状态在后续线路中的演化。

## 8. 理论推导与数值实验各自的作用

- **理论：** Born 定则给出测量结果的概率分布，是概率计算的物理依据。
- **数值实验：** 随机抽样帮助我们理解有限次数实验中的统计涨落，并检查代码是否正确实现了理论分布。

完成本题后，你应该能够区分量子态的复数振幅、测量概率、单次测量结果与重复测量的统计频率。下一步可以在此基础上学习测量后状态的投影与归一化。